# SpectraShift Week 9: freeze analysis contracts
Use CPU with Internet off. Attach source v8, Week 2 frozen, Week 5 contracts, Week 7 complete, Week 8 evaluation contracts, and Week 8 complete.


In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, sys, yaml

INPUT = Path('/kaggle/input')
projects = [p.parent for p in INPUT.rglob('pyproject.toml') if (p.parent / 'src/spectrashift/train/week9.py').is_file()]
if not projects:
    bundles = sorted(INPUT.rglob('spectrashift-kaggle-source.zip'))
    assert len(bundles) == 1, f'Expected one Week 9 source bundle, found {bundles}'
    source_work = Path('/tmp/spectrashift-week9-source')
    if source_work.exists(): shutil.rmtree(source_work)
    shutil.unpack_archive(str(bundles[0]), str(source_work))
    projects = [source_work]
assert projects, 'No Week 9 source tree found'
PROJECT = sorted(projects, key=lambda path: len(str(path)))[0]
sys.path.insert(0, str(PROJECT / 'src'))
os.chdir(PROJECT)

def unique_file(name):
    candidates = sorted(INPUT.rglob(name))
    by_hash = {}
    for path in candidates:
        by_hash.setdefault(hashlib.sha256(path.read_bytes()).hexdigest(), path)
    assert len(by_hash) == 1, f'Expected one unique {name}; found {candidates}'
    return next(iter(by_hash.values()))

WORK = Path('/kaggle/working/spectrashift-week9-contracts')
WORK.mkdir(parents=True, exist_ok=True)
MANIFEST = unique_file('partitions.parquet')
NORMALIZATION = unique_file('normalization.json')
STAGED = next(path.parent for path in INPUT.rglob('staging_summary.json'))
WEEK5_CONTRACTS = unique_file('week5_contracts_summary.json')
WEEK7_SUMMARY = unique_file('week7_run_summary.json')
LEDGER = unique_file('week8_checkpoint_ledger.csv')
WEEK8_SUMMARY = unique_file('week8_run_summary.json')
EVAL_LABELS = unique_file('evaluation_labels.parquet')
EVAL_CONTRACT = unique_file('evaluation_contract.json')
SUPPORT_CONTRACT = unique_file('support_contract.json')
config = yaml.safe_load((PROJECT / 'configs/analysis/week9.yaml').read_text())
config['paths'].update({
    'manifest_path': str(MANIFEST), 'staged_root': str(STAGED),
    'normalization_path': str(NORMALIZATION), 'week5_contracts_path': str(WEEK5_CONTRACTS),
    'week7_summary_path': str(WEEK7_SUMMARY), 'checkpoint_ledger_path': str(LEDGER),
    'week8_summary_path': str(WEEK8_SUMMARY), 'week8_output_dir': str(WEEK8_SUMMARY.parent),
    'evaluation_labels_path': str(EVAL_LABELS), 'evaluation_contract_path': str(EVAL_CONTRACT),
    'support_contract_path': str(SUPPORT_CONTRACT), 'contracts_output_dir': str(WORK),
})
RUNTIME_CONFIG = WORK / 'week9.yaml'
RUNTIME_CONFIG.write_text(yaml.safe_dump(config, sort_keys=False))
print({'project': str(PROJECT), 'work': str(WORK)})


In [ ]:
from spectrashift.train.week9 import freeze_week9_contracts
summary = freeze_week9_contracts(RUNTIME_CONFIG)
print(json.dumps(summary, indent=2))
assert summary['week9_contracts_complete']
assert summary['cka_patch_count'] == 1000
assert summary['nearest_neighbor_query_count'] == 100
assert summary['diagnostic_checkpoint_count'] == 6
assert summary['model_selection_after_week8'] is False
